# Trexquant — Earnings Return Prediction (Kaggle submission)

## Input
Attach **Earnings Return Prediction Challenge 2025Q4** in the notebook **Input** panel.

## Strategy
- **Walk-forward CV on `di`** with warmup + **embargo** (train only on past `di`).
- **Meta columns** are not z-scored or ranked.
- **Interactions**: 12 latent families (corr + missingness), pairwise ratio/diff/prod, **top 30 by full-train |Pearson|** with target — **optimistic for research**; for honest CV, interaction mining should be fold-local (see `advanced_experiments.py` / `next_stage.py`).
- **OOF Pearson** is computed on **covered** validation rows only (warmup rows excluded), not zero-filled.
- **CatBoost**; test prediction averaged over folds.

## Notes
- Do not use test labels. Interactions use only feature columns shared by train/test.
- Final model is fold-averaged OOF-style; for maximum leaderboard score you can add a second cell that retrains on full train (optional).

In [ ]:
import os
import gc
import json
import warnings
from glob import glob

import numpy as np
import pandas as pd
# Optional sklearn pieces (unused by default pipeline; handy for quick experiments)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## 1. Load Dataset

In [ ]:
def find_train_test():
    pairs = []
    for train_path in glob('/kaggle/input/**/train.csv', recursive=True):
        test_path = os.path.join(os.path.dirname(train_path), 'test.csv')
        if os.path.exists(test_path):
            pairs.append((train_path, test_path))
    if not pairs:
        raise FileNotFoundError(
            'No train.csv/test.csv found. Attach the competition dataset in the Kaggle Input pane first.'
        )
    return pairs[0]

train_path, test_path = find_train_test()
df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)

print('train shape:', df_train.shape)
print('test  shape:', df_test.shape)

## 2. Quick audit

- Train ~146k rows, test ~72k; many `f_*` columns have heavy missingness — CatBoost handles this.
- Use **walk-forward** splits (not random KFold) so validation `di` is always after training `di`.

In [ ]:
ID_COL     = 'id'
TARGET_COL = 'target'
TIME_COL   = 'di'

feature_cols = [c for c in df_train.columns if c not in [ID_COL, TARGET_COL]]

print('id unique train:', df_train[ID_COL].is_unique)
print('id unique test: ', df_test[ID_COL].is_unique)
print('di unique train:', df_train[TIME_COL].nunique())
print('si unique train:', df_train['si'].nunique())
print()
print('target describe:')
print(df_train[TARGET_COL].describe())
print('skew:', round(df_train[TARGET_COL].skew(), 4))
print('kurt:', round(df_train[TARGET_COL].kurt(), 4))

miss = df_train[feature_cols].isna().mean().sort_values(ascending=False)
print('\nTop 10 missing:')
print(miss.head(10).to_string())

## 3. Walk-forward validation (embargo)

Same logic as `baseline_experiments.walk_forward_time_splits`: train only on **past** `di` groups; drop the last **5** `di` groups before each validation block as embargo.

In [ ]:
import math

def pearson_corr(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    if y_true.size < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return float('nan')
    return float(np.corrcoef(y_true, y_pred)[0, 1])


def walk_forward_time_splits(time_series, n_splits=5, min_train_groups=252, embargo_groups=5):
    unique_di = np.sort(time_series.dropna().unique())
    U = len(unique_di)
    warmup = max(min_train_groups, math.ceil(0.30 * U))
    remaining = unique_di[warmup:]
    chunks = np.array_split(remaining, n_splits)
    idx = np.arange(len(time_series))
    tv = time_series.to_numpy()
    splits = []
    for chunk in chunks:
        if len(chunk) == 0:
            continue
        valid_start = chunk[0]
        pre = unique_di[unique_di < valid_start]
        allowed = pre[:-embargo_groups] if embargo_groups > 0 and len(pre) > embargo_groups else np.array([], dtype=pre.dtype)
        tr_idx = idx[np.isin(tv, allowed)]
        va_idx = idx[np.isin(tv, chunk)]
        if len(tr_idx) > 0 and len(va_idx) > 0:
            splits.append((tr_idx, va_idx))
    if len(splits) < 2:
        raise ValueError('Too few walk-forward folds')
    return splits


N_SPLITS = 5
EMBARGO = 5
MIN_TRAIN_GROUPS = 252
splits = walk_forward_time_splits(
    df_train[TIME_COL], n_splits=N_SPLITS,
    min_train_groups=MIN_TRAIN_GROUPS, embargo_groups=EMBARGO,
)
print(f'Walk-forward folds (embargo={EMBARGO} di groups):')
for i, (tr, va) in enumerate(splits):
    tr_di = df_train.iloc[tr][TIME_COL]
    va_di = df_train.iloc[va][TIME_COL]
    ok = tr_di.max() < va_di.min()
    print(f'  fold {i}: train di [{tr_di.min()}–{tr_di.max()}]  valid [{va_di.min()}–{va_di.max()}]  '
          f'n_train={len(tr):,} n_valid={len(va):,}  embargo_ok={ok}')

## 4. Latent-family interactions + CatBoost

1. Cluster **172** `f_*` columns into **12** families (0.5 × distance from |corr| + 0.5 × distance from missingness pattern correlation), same as `next_stage.py`.
2. Pick one **representative** per family (highest |Pearson| with `target`).
3. For every pair of representatives, score **ratio / diff / prod** vs `target`; materialise **top 30** (clip train 1–99%, apply same bounds to test).
4. **CatBoost** on `meta + all f_* + interaction columns`, walk-forward CV, average test preds across folds.

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from catboost import CatBoostRegressor

STOCK_COL = 'si'
META_CANDIDATES = ['si', 'di', 'industry', 'sector', 'top2000', 'top1000', 'top500']
N_FAMILIES = 12
IX_THRESHOLD = 0.015
TOP_IX = 30

meta_cols = [c for c in META_CANDIDATES if c in df_train.columns]
anon_cols = sorted(c for c in df_train.columns if c.startswith('f_'))
y_full = df_train[TARGET_COL].to_numpy(np.float64)
X = df_train[anon_cols].to_numpy(np.float64)
n = len(anon_cols)

# Pairwise |corr| distance between anonymous features
print('Clustering anonymous features into latent families...')
corr_matrix = np.eye(n)
for i in range(n):
    for j in range(i + 1, n):
        xi, xj = X[:, i], X[:, j]
        m = np.isfinite(xi) & np.isfinite(xj)
        if m.sum() < 50:
            c = 0.0
        else:
            c = float(np.corrcoef(xi[m], xj[m])[0, 1])
            if not np.isfinite(c):
                c = 0.0
        corr_matrix[i, j] = corr_matrix[j, i] = c
dist_corr = 1.0 - np.abs(corr_matrix)

miss = (~np.isfinite(X)).astype(np.float64)
ms = miss.std(axis=0)
miss_corr = np.eye(n)
for i in range(n):
    for j in range(i + 1, n):
        if ms[i] == 0 or ms[j] == 0:
            c = 1.0 if (miss[:, i] == miss[:, j]).all() else 0.0
        else:
            c = float(np.corrcoef(miss[:, i], miss[:, j])[0, 1])
            if not np.isfinite(c):
                c = 0.0
        miss_corr[i, j] = miss_corr[j, i] = c
dist_miss = 1.0 - np.abs(miss_corr)

dist_combined = 0.5 * dist_corr + 0.5 * dist_miss
np.fill_diagonal(dist_combined, 0.0)
Z = linkage(squareform(dist_combined, checks=False), method='ward')
family_labels = fcluster(Z, N_FAMILIES, criterion='maxclust')

feat_corrs = []
for i, col in enumerate(anon_cols):
    xi = X[:, i]
    m = np.isfinite(xi) & np.isfinite(y_full)
    if m.sum() < 50:
        feat_corrs.append(0.0)
    else:
        c = float(np.corrcoef(xi[m], y_full[m])[0, 1])
        feat_corrs.append(0.0 if not np.isfinite(c) else c)
feat_corrs = np.array(feat_corrs)

family_reps = {}
for fam in range(1, N_FAMILIES + 1):
    idx = np.where(family_labels == fam)[0]
    best_local = idx[np.argmax(np.abs(feat_corrs[idx]))]
    family_reps[fam] = anon_cols[best_local]
reps = list(family_reps.values())
print(f'Family representatives ({len(reps)}): {reps}')

# Candidate interactions (same rule as next_stage.py)
candidates = []
for i in range(len(reps)):
    for j in range(i + 1, len(reps)):
        ri, rj = reps[i], reps[j]
        xi = df_train[ri].to_numpy(np.float64)
        xj = df_train[rj].to_numpy(np.float64)
        mask = np.isfinite(xi) & np.isfinite(xj) & np.isfinite(y_full)
        if mask.sum() < 100:
            continue
        ratio = xi / (np.abs(xj) + 1e-6)
        rm = ratio[mask]
        rm = np.clip(rm, np.nanpercentile(rm, 1), np.nanpercentile(rm, 99))
        diff = xi - xj
        prod = xi * xj
        for kind, arr, c in [
            ('ratio', ratio, pearson_corr(y_full[mask], rm)),
            ('diff', diff, pearson_corr(y_full[mask], diff[mask])),
            ('prod', prod, pearson_corr(y_full[mask], prod[mask])),
        ]:
            if np.isfinite(c) and abs(c) > IX_THRESHOLD:
                name = f'{ri}_x_{rj}_{kind}'
                candidates.append({'name': name, 'f_i': ri, 'f_j': rj, 'kind': kind, 'abs_corr': abs(c)})

candidates = sorted(candidates, key=lambda r: r['abs_corr'], reverse=True)
top_ix = candidates[:TOP_IX]
print(f'Interaction candidates |r|>{IX_THRESHOLD}: {len(candidates)}  (using top {len(top_ix)})')

df_tr = df_train.copy()
df_te = df_test.copy()
ix_names = []
for row in top_ix:
    ri, rj, kind, name = row['f_i'], row['f_j'], row['kind'], row['name']
    xi_tr = df_tr[ri].to_numpy(np.float64)
    xj_tr = df_tr[rj].to_numpy(np.float64)
    xi_te = df_te[ri].to_numpy(np.float64)
    xj_te = df_te[rj].to_numpy(np.float64)
    if kind == 'ratio':
        tr_val = xi_tr / (np.abs(xj_tr) + 1e-6)
        te_val = xi_te / (np.abs(xj_te) + 1e-6)
    elif kind == 'diff':
        tr_val = xi_tr - xj_tr
        te_val = xi_te - xj_te
    else:
        tr_val = xi_tr * xj_tr
        te_val = xi_te * xj_te
    lo, hi = np.nanpercentile(tr_val[np.isfinite(tr_val)], [1, 99])
    df_tr[name] = np.clip(tr_val, lo, hi)
    df_te[name] = np.clip(te_val, lo, hi)
    ix_names.append(name)

base_feats = [c for c in df_train.columns if c not in [ID_COL, TARGET_COL]]
feature_cols = base_feats + ix_names
feature_cols = [c for c in feature_cols if c in df_tr.columns and c in df_te.columns]

cat_cols = [c for c in meta_cols if c in feature_cols and c != TIME_COL]
train_cb = df_tr[feature_cols].copy()
test_cb = df_te[feature_cols].copy()
for c in cat_cols:
    train_cb[c] = train_cb[c].astype(str)
    test_cb[c] = test_cb[c].astype(str)

y = y_full
oof_pred = np.zeros(len(df_train), dtype=np.float64)
test_pred = np.zeros(len(df_test), dtype=np.float64)

oof_covered = np.zeros(len(df_train), dtype=bool)
for fold_id, (tr_idx, va_idx) in enumerate(splits):
    tr_di = df_train.iloc[tr_idx][TIME_COL]
    va_di = df_train.iloc[va_idx][TIME_COL]
    assert tr_di.max() < va_di.min(), 'walk-forward violated'

    model = CatBoostRegressor(
        loss_function='RMSE',
        eval_metric='RMSE',
        depth=6,
        learning_rate=0.05,
        iterations=800,
        l2_leaf_reg=3.0,
        random_seed=42 + fold_id,
        subsample=0.8,
        bootstrap_type='Bernoulli',
        od_type='Iter',
        od_wait=80,
        verbose=False,
    )
    model.fit(
        train_cb.iloc[tr_idx], y[tr_idx],
        cat_features=cat_cols if cat_cols else None,
        eval_set=(train_cb.iloc[va_idx], y[va_idx]),
        use_best_model=True,
    )
    oof_pred[va_idx] = model.predict(train_cb.iloc[va_idx])
    oof_covered[va_idx] = True
    test_pred += model.predict(test_cb) / len(splits)
    fold_corr = pearson_corr(y[va_idx], oof_pred[va_idx])
    print(f'Fold {fold_id}: Pearson={fold_corr:.6f}  best_iter={model.best_iteration_}')
    del model
    gc.collect()

cov_frac = float(oof_covered.mean())
if oof_covered.sum() >= 2:
    oof_corr = pearson_corr(y[oof_covered], oof_pred[oof_covered])
else:
    oof_corr = float('nan')
print(f'\nOOF Pearson (covered rows only; warmup excluded): {oof_corr:.6f}  coverage_frac={cov_frac:.4f}')
print(f'Features used: {len(feature_cols)} (incl. {len(ix_names)} interactions)')

## 5. Submission

**Requirements:**
- Must contain `"target"` column.
- No `nan` or `inf` in `"target"`.
- At least `10%` of target values must be non-zero.

In [ ]:
# Mild post-processing: clip extreme outliers
lo, hi = np.nanpercentile(test_pred, [0.5, 99.5])
test_pred = np.clip(test_pred, lo, hi)

# Guarantee at least 10% non-zero
if (np.abs(test_pred) > 0).mean() < 0.10:
    test_pred = test_pred + 1e-9

df_submission = pd.DataFrame({
    'id':     df_test[ID_COL],
    'target': test_pred,
})

assert np.isfinite(df_submission['target']).all(), 'Non-finite predictions found'
assert (df_submission['target'].abs() > 0).mean() >= 0.10, 'Less than 10% non-zero'

filename = '/kaggle/working/submission.csv'
df_submission.to_csv(filename, index=False)

print(f'Saved: {filename}')
print(f'Rows: {len(df_submission):,}')
print(f'Non-zero fraction: {(df_submission["target"].abs() > 0).mean():.4f}')
print(f'OOF Pearson (walk-forward + interactions, local estimate): {oof_corr:.6f}')
print(df_submission.head())